In [ ]:
# tests/test_solution.py
import pytest
from solution import Pointer, PointerError

# --- Tests for Prompt 1 ---

def test_pointer_initialization():
    """Tests valid and invalid pointer string formats."""
    # Valid
    Pointer("")
    Pointer("/foo")
    Pointer("/foo/0")
    Pointer("/")
    Pointer("/a~1b")
    Pointer("/c%d")
    Pointer("/e^f")
    Pointer("/g|h")
    Pointer("/i\\j")
    Pointer("/k\"l")
    Pointer("/ ")
    Pointer("/m~0n")

    # Invalid (must not start with anything but / or be empty)
    with pytest.raises(PointerError):
        Pointer("foo")
    with pytest.raises(PointerError):
        Pointer("foo/")
    with pytest.raises(PointerError):
        Pointer("/foo/~") # Incomplete escape
    with pytest.raises(PointerError):
        Pointer("/foo/~2") # Invalid escape

def test_pointer_resolve_success():
    """Tests successful resolution in various document structures."""
    doc = {
        "foo": ["bar", "baz"],
        "": 0,
        "a/b": 1,
        "c%d": 2,
        "e^f": 3,
        "g|h": 4,
        "i\\j": 5,
        "k\"l": 6,
        " ": 7,
        "m~n": 8
    }
    assert Pointer("").resolve(doc) == doc
    assert Pointer("/foo").resolve(doc) == ["bar", "baz"]
    assert Pointer("/foo/0").resolve(doc) == "bar"
    assert Pointer("/").resolve(doc) == 0
    assert Pointer("/a~1b").resolve(doc) == 1
    assert Pointer("/c%d").resolve(doc) == 2
    assert Pointer("/e^f").resolve(doc) == 3
    assert Pointer("/g|h").resolve(doc) == 4
    assert Pointer("/i\\j").resolve(doc) == 5
    assert Pointer("/k\"l").resolve(doc) == 6
    assert Pointer("/ ").resolve(doc) == 7
    assert Pointer("/m~0n").resolve(doc) == 8

def test_pointer_resolve_array_edge_cases():
    """Tests array-specific resolution."""
    doc = [0, 1, 2]
    assert Pointer("/0").resolve(doc) == 0
    assert Pointer("/2").resolve(doc) == 2
    # According to RFC, '-' is not valid for resolution, only for 'add' op
    with pytest.raises(PointerError):
        Pointer("/-").resolve(doc)

def test_pointer_resolve_errors():
    """Tests resolution failures for non-existent paths."""
    doc = {"foo": "bar"}
    # Key not found in object
    with pytest.raises(PointerError):
        Pointer("/baz").resolve(doc)
    # Index out of bounds in array
    with pytest.raises(PointerError):
        Pointer("/foo/0").resolve(doc) # /foo is not an array
    
    doc_list = [1, 2, 3]
    with pytest.raises(PointerError):
        Pointer("/3").resolve(doc_list) # out of bounds
    with pytest.raises(PointerError):
        Pointer("/-1").resolve(doc_list) # negative index invalid
    with pytest.raises(PointerError):
        Pointer("/foo").resolve(doc_list) # not an object

    # Nested resolution failure
    doc_nested = {"a": {"b": 1}}
    with pytest.raises(PointerError):
        Pointer("/a/c").resolve(doc_nested)

def test_pointer_rfc_examples():
    """Tests examples directly from RFC 6901."""
    doc = {
        "foo": ["bar", "baz"],
        "": 0,
        "a/b": 1,
        "c%d": 2,
        "e^f": 3,
        "g|h": 4,
        "i\\j": 5,
        "k\"l": 6,
        " ": 7,
        "m~n": 8
    }
    assert Pointer("").resolve(doc) == doc
    assert Pointer("/foo").resolve(doc) == ["bar", "baz"]
    assert Pointer("/foo/0").resolve(doc) == "bar"
    assert Pointer("/").resolve(doc) == 0
    assert Pointer("/a~1b").resolve(doc) == 1
    assert Pointer("/c%d").resolve(doc) == 2
    assert Pointer("/e^f").resolve(doc) == 3
    assert Pointer("/g|h").resolve(doc) == 4
    assert Pointer("/i\\j").resolve(doc) == 5
    assert Pointer("/k\"l").resolve(doc) == 6
    assert Pointer("/ ").resolve(doc) == 7
    assert Pointer("/m~0n").resolve(doc) == 8

In [ ]:
# --- Tests for Prompt 2 ---
from solution import apply_patch, PatchError
import copy

def test_apply_patch_add():
    doc = {"foo": "bar"}
    patch = [{"op": "add", "path": "/baz", "value": "qux"}]
    new_doc = apply_patch(doc, patch)
    assert new_doc == {"foo": "bar", "baz": "qux"}
    assert doc == {"foo": "bar"} # Original unmodified

    doc = {"foo": ["bar"]}
    patch = [{"op": "add", "path": "/foo/1", "value": "qux"}]
    new_doc = apply_patch(doc, patch)
    assert new_doc == {"foo": ["bar", "qux"]}
    
    # Add to end of array with '-'
    patch_append = [{"op": "add", "path": "/foo/-", "value": "last"}]
    new_doc_append = apply_patch(doc, patch_append)
    assert new_doc_append == {"foo": ["bar", "last"]}


def test_apply_patch_remove():
    doc = {"foo": "bar", "baz": "qux"}
    patch = [{"op": "remove", "path": "/baz"}]
    new_doc = apply_patch(doc, patch)
    assert new_doc == {"foo": "bar"}
    assert doc == {"foo": "bar", "baz": "qux"}

    doc_list = {"foo": ["bar", "qux", "baz"]}
    patch = [{"op": "remove", "path": "/foo/1"}]
    new_doc = apply_patch(doc_list, patch)
    assert new_doc == {"foo": ["bar", "baz"]}


def test_apply_patch_replace():
    doc = {"foo": "bar", "baz": "qux"}
    patch = [{"op": "replace", "path": "/baz", "value": "boo"}]
    new_doc = apply_patch(doc, patch)
    assert new_doc == {"foo": "bar", "baz": "boo"}
    assert doc == {"foo": "bar", "baz": "qux"}


def test_apply_patch_multiple_ops():
    doc = {"foo": "bar"}
    patch = [
        {"op": "add", "path": "/baz", "value": [1, 2, 3]},
        {"op": "replace", "path": "/foo", "value": "qux"},
        {"op": "remove", "path": "/baz/1"},
    ]
    new_doc = apply_patch(doc, patch)
    assert new_doc == {"foo": "qux", "baz": [1, 3]}


def test_apply_patch_atomicity_and_immutability():
    doc = {"foo": "bar"}
    # Last op fails
    patch = [
        {"op": "add", "path": "/baz", "value": "qux"},
        {"op": "remove", "path": "/nonexistent"},
    ]
    original_doc = copy.deepcopy(doc)
    with pytest.raises(PatchError):
        apply_patch(doc, patch)
    assert doc == original_doc, "Original document was modified"


def test_apply_patch_errors():
    # Remove non-existent path
    with pytest.raises(PatchError):
        apply_patch({"foo": "bar"}, [{"op": "remove", "path": "/baz"}])
    # Replace non-existent path
    with pytest.raises(PatchError):
        apply_patch({"foo": "bar"}, [{"op": "replace", "path": "/baz", "value": 1}])
    # Add to a nested path where parent does not exist
    with pytest.raises(PatchError):
        apply_patch({"foo": "bar"}, [{"op": "add", "path": "/baz/qux", "value": 1}])
    # Invalid op
    with pytest.raises(PatchError):
        apply_patch({"foo": "bar"}, [{"op": "fly", "path": "/foo"}])
    # Missing path
    with pytest.raises(PatchError):
        apply_patch({"foo": "bar"}, [{"op": "add", "value": 1}])


def test_rfc6902_examples():
    doc = { "foo": "bar"}
    patch = [{ "op": "add", "path": "/baz", "value": "qux" }]
    new_doc = apply_patch(doc, patch)
    assert new_doc == { "baz": "qux", "foo": "bar" }

    doc = { "foo": [ "bar", "baz" ] }
    patch = [{ "op": "add", "path": "/foo/1", "value": "qux" }]
    new_doc = apply_patch(doc, patch)
    assert new_doc == { "foo": [ "bar", "qux", "baz" ] }

    doc = { "baz": "qux", "foo": "bar" }
    patch = [{ "op": "remove", "path": "/baz" }]
    new_doc = apply_patch(doc, patch)
    assert new_doc == { "foo": "bar" }

    doc = { "foo": [ "bar", "qux", "baz" ] }
    patch = [{ "op": "remove", "path": "/foo/1" }]
    new_doc = apply_patch(doc, patch)
    assert new_doc == { "foo": [ "bar", "baz" ] }
    
    doc = { "baz": "qux", "foo": "bar" }
    patch = [{ "op": "replace", "path": "/baz", "value": "boo" }]
    new_doc = apply_patch(doc, patch)
    assert new_doc == { "foo": "bar", "baz": "boo" }

In [ ]:
# --- Tests for Prompt 3 ---
from solution import create_patch

def test_create_patch_flat_objects():
    # Add
    doc1 = {"a": 1}
    doc2 = {"a": 1, "b": 2}
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    assert patch == [{"op": "add", "path": "/b", "value": 2}]

    # Remove
    doc1 = {"a": 1, "b": 2}
    doc2 = {"a": 1}
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    assert patch == [{"op": "remove", "path": "/b"}]

    # Replace
    doc1 = {"a": 1}
    doc2 = {"a": 2}
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    assert patch == [{"op": "replace", "path": "/a", "value": 2}]

    # No change
    doc1 = {"a": 1}
    doc2 = {"a": 1}
    patch = create_patch(doc1, doc2)
    assert patch == []
    assert apply_patch(doc1, patch) == doc2


def test_create_patch_combination_and_determinism():
    doc1 = {"a": 1, "b": 2, "c": 3}
    doc2 = {"b": 20, "c": 3, "d": 4}
    # Expected: remove a, replace b, add d
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    
    # Check for determinism (keys should be sorted)
    expected_patch = [
        {"op": "remove", "path": "/a"},
        {"op": "replace", "path": "/b", "value": 20},
        {"op": "add", "path": "/d", "value": 4},
    ]
    # We allow any order that produces the right result, but a sorted one is best.
    # Let's test by applying. A more strict test would sort and compare.
    reconstructed_doc = apply_patch(doc1, patch)
    assert reconstructed_doc == doc2
    assert len(patch) == 3




In [ ]:
# --- Tests for Prompt 4 ---

def test_create_patch_lcs_array_simple():
    # Simple insert
    doc1 = [1, 2, 3, 4]
    doc2 = [1, 2, 5, 3, 4]
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    # A naive diff would produce 3 ops, a good one just one.
    assert len(patch) == 1
    assert patch[0]["op"] == "add"
    assert patch[0]["path"] == "/2"
    assert patch[0]["value"] == 5

    # Simple delete
    doc1 = ["a", "b", "c"]
    doc2 = ["a", "c"]
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    assert len(patch) == 1
    assert patch[0] == {"op": "remove", "path": "/1"}


def test_create_patch_lcs_array_complex():
    doc1 = [1, 2, 3, 4, 5]
    doc2 = [0, 1, 3, 4, 6, 5]
    patch = create_patch(doc1, doc2)
    applied = apply_patch(doc1, patch)
    assert applied == doc2

    # A good patch would be: remove 2, add 0, add 6.
    # The exact patch depends on the LCS implementation, but it should be minimal.
    # A naive replace would be 1 op. A good diff is >1 but < (len1+len2)
    assert len(patch) < 6
    assert any(p['op'] == 'remove' for p in patch)
    assert any(p['op'] == 'add' and p['value'] == 0 for p in patch)
    assert any(p['op'] == 'add' and p['value'] == 6 for p in patch)


def test_create_patch_lcs_nested():
    doc1 = {"data": [
        {"id": 1, "value": "A"},
        {"id": 2, "value": "B"},
        {"id": 3, "value": "C"},
    ]}
    doc2 = {"data": [
        {"id": 1, "value": "A"},
        {"id": 4, "value": "D"}, # Added
        {"id": 3, "value": "C-mod"}, # Modified
    ]}

    patch = create_patch(doc1, doc2)
    applied = apply_patch(doc1, patch)
    assert applied == doc2
    # Expected: remove /data/1, add /data/1, replace /data/2/value (now at index 2)
    # The exact sequence depends on implementation, but apply_patch confirms correctness.
    # The key is that it does NOT replace the whole 'data' array.
    assert not any(p['path'] == '/data' and p['op'] == 'replace' for p in patch)
    assert any(p['path'] == '/data/1' and p['op'] == 'remove' for p in patch)
    assert any(p['path'] == '/data/2/value' for p in patch) # Note: path is post-removal of item at index 1



def test_create_patch_with_non_flat_objects_and_arrays():
    # Add nested object
    doc1 = {"a": 1}
    doc2 = {"a": 1, "b": {"c": 2}}
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    assert patch == [{"op": "add", "path": "/b", "value": {"c": 2}}]
    
    # Replace with array
    doc1 = {"a": 1}
    doc2 = {"a": [1, 2]}
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    assert patch == [{"op": "replace", "path": "/a", "value": [1, 2]}]

    # Modify nested object value
    doc1 = {"a": {"b": 1}}
    doc2 = {"a": {"b": 2}}
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    assert patch == [{"op": "replace", "path": "/a/b", "value": 2}]

In [ ]:
# --- Tests for Prompt 5 ---

def test_apply_patch_move_and_copy():
    # Move
    doc = {"foo": {"bar": "baz", "waldo": "fred"}, "qux": {"corge": "grault"}}
    patch = [{"op": "move", "from": "/foo/waldo", "path": "/qux/thud"}]
    new_doc = apply_patch(doc, patch)
    assert new_doc == {"foo": {"bar": "baz"}, "qux": {"corge": "grault", "thud": "fred"}}
    assert "waldo" not in new_doc["foo"]

    # Copy
    doc = {"foo": "bar", "baz": "qux"}
    patch = [{"op": "copy", "from": "/foo", "path": "/waldo"}]
    new_doc = apply_patch(doc, patch)
    assert new_doc == {"foo": "bar", "baz": "qux", "waldo": "bar"}
    assert doc == {"foo": "bar", "baz": "qux"} # Original unmodified


def test_apply_patch_move_and_copy_errors():
    doc = {"foo": "bar"}
    # 'from' path does not exist
    with pytest.raises(PatchError):
        apply_patch(doc, [{"op": "move", "from": "/baz", "path": "/qux"}])
    with pytest.raises(PatchError):
        apply_patch(doc, [{"op": "copy", "from": "/baz", "path": "/qux"}])
    # Move a value into its own child (invalid)
    doc_nested = {"foo": {"bar": 1}}
    with pytest.raises(PatchError):
        apply_patch(doc_nested, [{"op": "move", "from": "/foo", "path": "/foo/bar"}])


def test_create_patch_detects_move():
    # Detects moved object in an array
    doc1 = [{"id": 1}, {"id": 2}, {"id": 3}]
    doc2 = [{"id": 3}, {"id": 1}, {"id": 2}]
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    # An optimal patch is two moves. A simple one is also fine.
    # Must be better than 3 removes and 3 adds.
    assert len(patch) <= 2
    assert all(p["op"] == "move" for p in patch)


def test_create_patch_detects_copy():
    # Detects copied value
    doc1 = {"a": 1, "b": 2}
    doc2 = {"a": 1, "b": 2, "c": 1}
    patch = create_patch(doc1, doc2)
    assert apply_patch(doc1, patch) == doc2
    # Should be a 'copy', not an 'add' with a literal value.
    assert patch == [{"op": "copy", "from": "/a", "path": "/c"}]

In [ ]:
# --- Tests for Prompt 6 ---
from solution import create_inverse

def test_apply_patch_test_operation():
    doc = {"a": 1, "b": [10, 20]}
    # Successful test
    patch_success = [
        {"op": "test", "path": "/a", "value": 1},
        {"op": "add", "path": "/c", "value": 3}
    ]
    new_doc = apply_patch(doc, patch_success)
    assert new_doc == {"a": 1, "b": [10, 20], "c": 3}

    # Failed test
    patch_fail = [
        {"op": "test", "path": "/a", "value": 999}, # This fails
        {"op": "add", "path": "/c", "value": 3}
    ]
    with pytest.raises(PatchError):
        apply_patch(doc, patch_fail)


def test_create_inverse_patch():
    # Simple add/remove/replace
    doc = {"a": 1, "b": "hello"}
    patch = [
        {"op": "add", "path": "/c", "value": [1,2]},
        {"op": "remove", "path": "/a"},
        {"op": "replace", "path": "/b", "value": "world"},
    ]
    inverse_patch = create_inverse(doc, patch)
    
    # Apply original patch, then inverse, should get back original doc
    intermediate_doc = apply_patch(doc, patch)
    final_doc = apply_patch(intermediate_doc, inverse_patch)
    assert final_doc == doc

def test_create_inverse_move_copy_test():
    doc = {"a": [1, 2], "b": 3}
    patch = [
        {"op": "test", "path": "/b", "value": 3},
        {"op": "move", "from": "/a", "path": "/c"},
        {"op": "copy", "from": "/c/0", "path": "/d"},
    ]
    inverse_patch = create_inverse(doc, patch)
    
    intermediate_doc = apply_patch(doc, patch)
    final_doc = apply_patch(intermediate_doc, inverse_patch)
    assert final_doc == doc

def test_inverse_of_inverse():
    doc = {"a": 1, "b": 2}
    patch = [{"op": "add", "path": "/c", "value": 3}, {"op": "remove", "path": "/a"}]
    
    intermediate = apply_patch(doc, patch)
    inverse1 = create_inverse(doc, patch)
    
    # The inverse of the inverse should be the original patch (semantically)
    inverse2 = create_inverse(intermediate, inverse1)
    
    final = apply_patch(intermediate, inverse1)
    assert final == doc

    assert apply_patch(doc, inverse2) == intermediate

In [ ]:
# --- Tests for Prompt 7 ---
from solution import merge, MergeError

def test_merge_non_conflicting():
    # Both add different keys
    base = {"a": 1}
    doc1 = {"a": 1, "b": 2}
    doc2 = {"a": 1, "c": 3}
    result = merge(base, doc1, doc2)
    assert result == {"a": 1, "b": 2, "c": 3}

    # One modifies, one adds
    base = {"a": 1}
    doc1 = {"a": 10}
    doc2 = {"a": 1, "b": 2}
    result = merge(base, doc1, doc2)
    assert result == {"a": 10, "b": 2}

    # Both make the same change
    base = {"a": 1}
    doc1 = {"a": 2}
    doc2 = {"a": 2}
    result = merge(base, doc1, doc2)
    assert result == {"a": 2}


def test_merge_array_non_conflicting():
    base = {"items": [1, 2, 3]}
    doc1 = {"items": [1, 2, 3, 4]} # Add to end
    doc2 = {"items": [0, 1, 2, 3]} # Add to start
    result = merge(base, doc1, doc2)
    assert result == {"items": [0, 1, 2, 3, 4]}


def test_merge_conflicting_raises_error():
    # Both modify the same value differently
    base = {"a": 1}
    doc1 = {"a": 2}
    doc2 = {"a": 3}
    with pytest.raises(MergeError):
        merge(base, doc1, doc2)

    # One modifies, one deletes
    base = {"a": 1, "b": 2}
    doc1 = {"a": 10, "b": 2}
    doc2 = {"a": 1}
    with pytest.raises(MergeError):
        merge(base, doc1, doc2)

In [ ]:
# --- Tests for Prompt 8 ---

def test_merge_with_conflict_reporting():
    # Simple value conflict
    base = {"a": 1, "b": 10}
    doc1 = {"a": 2, "b": 10}
    doc2 = {"a": 3, "b": 10}
    
    result, conflicts = merge(base, doc1, doc2)
    assert result == {"a": 1, "b": 10} # 'a' reverts to base
    assert len(conflicts) == 1
    conflict = conflicts[0]
    assert conflict["path"] == "/a"
    assert "doc1_value" in conflict and "doc2_value" in conflict


def test_merge_delete_modify_conflict():
    base = {"a": 1, "b": 2}
    doc1 = {"a": 10, "b": 2} # Modify a
    doc2 = {"b": 2}          # Delete a
    
    result, conflicts = merge(base, doc1, doc2)
    assert result == {"a": 1, "b": 2} # Reverts to base
    assert len(conflicts) == 1
    assert conflicts[0]["path"] == "/a"


def test_merge_mixed_conflicts_and_success():
    base = {"a": 1, "b": 2, "c": 3, "d": 4}
    doc1 = {"a": 10, "b": 2, "c": 3, "d": 4, "e": 5} # Modify a, Add e
    doc2 = {"a": 1, "b": 20, "c": 3, "d": 4}       # Modify b
    
    result, conflicts = merge(base, doc1, doc2)
    # Successful merges should still be present
    assert result == {"a": 1, "b": 20, "c": 3, "d": 4, "e": 5}
    assert conflicts == [] # This scenario is actually non-conflicting
    
    # Now, a real mixed case
    doc2_conflict = {"a": 11, "b": 20, "c": 3, "d": 4}
    result, conflicts = merge(base, doc1, doc2_conflict)
    assert result['b'] == 20 # non-conflict
    assert result['e'] == 5  # non-conflict
    assert result['a'] == 1  # conflict, reverted
    assert len(conflicts) == 1
    assert conflicts[0]['path'] == '/a'

In [ ]:
# --- Tests for Prompt 9 ---
from solution import transform_op

def test_transform_op_independent():
    op1 = {"op": "add", "path": "/a", "value": 1}
    op2 = {"op": "add", "path": "/b", "value": 2}
    op1_p, op2_p = transform_op(op1, op2)
    assert op1_p == op1
    assert op2_p == op2


def test_transform_op_array_insert():
    # Concurrent inserts, op2 comes first
    op1 = {"op": "add", "path": "/0", "value": "a"}
    op2 = {"op": "add", "path": "/0", "value": "b"}
    op1_p, op2_p = transform_op(op1, op2)
    # If op2 runs first, op1's path must be shifted
    assert op1_p == {"op": "add", "path": "/1", "value": "a"}
    # If op1 runs first, op2's path must be shifted
    assert op2_p == {"op": "add", "path": "/1", "value": "b"}

    # Test convergence
    base = []
    doc_1_first = apply_patch(apply_patch(base, op1), op2_p)
    doc_2_first = apply_patch(apply_patch(base, op2), op1_p)
    assert doc_1_first == doc_2_first
    assert doc_1_first in (["a", "b"], ["b", "a"]) # Order can differ, but must be consistent


def test_transform_op_array_remove():
    op1 = {"op": "remove", "path": "/1"}
    op2 = {"op": "remove", "path": "/3"}
    # op2's index is after op1's
    op1_p, op2_p = transform_op(op1, op2)
    # op1 doesn't change
    assert op1_p == op1
    # op2's index must be shifted
    assert op2_p["path"] == "/2"

    # Test convergence
    base = [0, 1, 2, 3, 4]
    doc_1_first = apply_patch(apply_patch(base, op1), op2_p)
    doc_2_first = apply_patch(apply_patch(base, op2), op1_p)
    assert doc_1_first == doc_2_first
    assert doc_1_first == [0, 2, 4]

In [ ]:
# --- Tests for Prompt 10 ---
from solution import transform_patch

def test_transform_patch_convergence_property():
    base = {"text": "Hello ", "items": [1, 2, 3]}
    
    # User 1 adds to text and removes an item
    patch1 = [
        {"op": "replace", "path": "/text", "value": "Hello World"},
        {"op": "remove", "path": "/items/1"}
    ]
    # User 2 adds an item at the beginning and also modifies text
    patch2 = [
        {"op": "add", "path": "/items/0", "value": 0},
        {"op": "replace", "path": "/text", "value": "Hello!"}
    ]

    patch1_p, patch2_p = transform_patch(patch1, patch2)

    # Apply P1, then transformed P2
    result1 = apply_patch(apply_patch(base, patch1), patch2_p)

    # Apply P2, then transformed P1
    result2 = apply_patch(apply_patch(base, patch2), patch1_p)

    # The final state must be identical, regardless of application order
    assert result1 == result2

    # A simple check on the likely transformed state (assuming P1 wins text conflict)
    # The text conflict resolution depends on the transform_op implementation.
    # A common strategy is "last write wins" or "op1 wins".
    # Assuming op1 wins, text should be "Hello World".
    assert result1["text"] == "Hello World"
    # Items should be [0, 1, 3]
    assert result1["items"] == [0, 1, 3]

def test_transform_patch_complex_array_shuffle():
    base = [0, 1, 2, 3, 4, 5, 6]
    # P1 moves an element forward
    patch1 = [{"op": "move", "from": "/1", "path": "/4"}] # [0, 2, 3, 4, 1, 5, 6]
    # P2 deletes elements around the move
    patch2 = [{"op": "remove", "path": "/0"}, {"op": "remove", "path": "/5"}] # [1, 2, 3, 4, 6]

    patch1_p, patch2_p = transform_patch(patch1, patch2)
    
    result1 = apply_patch(apply_patch(base, patch1), patch2_p)
    result2 = apply_patch(apply_patch(base, patch2), patch1_p)

    assert result1 == result2
    # Expected final state: [2, 3, 4, 1, 6]
    assert result1 == [2, 3, 4, 1, 6]